In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()
if os.environ['GOOGLE_API_KEY']:
    print("API key is set.")
else:
    raise ValueError("API key is not set")

API key is set.


In [2]:
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.tools import tool

In [18]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.7-flash")

In [19]:
from typing import Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class graph_schema(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)

    

In [20]:
from langchain_community.tools import DuckDuckGoSearchRun
@tool
def duckduckgo(query: str) -> str:
    """this tool is use to search anything on internet for research and rescent events"""
    search = DuckDuckGoSearchRun()
    response = search.invoke(query)
    return response

In [21]:
import requests
@tool
def weather(city: str) -> str:
    """use this tools if you want to retrieve weather of any city"""
    city
    response = requests.get(f"https://wttr.in/{city}?format=j1")
    return response.text

In [22]:
tools = [duckduckgo,weather]
llm_with_tools = llm.bind_tools(tools)

In [23]:
def agent_node(state: graph_schema) -> dict:
    response = llm_with_tools.invoke(state.messages)
    return {"messages": [response]}

In [24]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image,display
from langgraph.prebuilt import ToolNode,tools_condition

builder = StateGraph(graph_schema)

builder.add_node("agent_node", agent_node)
builder.add_node("tools",ToolNode(tools))

builder.add_edge(START, "agent_node")
builder.add_conditional_edges(
    "agent_node",tools_condition
)

builder.add_edge("tools","agent_node")
checkpointer = MemorySaver()
graph = builder.compile(checkpointer = checkpointer)

config = { "configurable": { "thread_id": "run-1"}}
input_query = {"messages": [
        SystemMessage(content="If a tool can do the task, call it instead of answering yourself."),
        HumanMessage(content="what just happened in nepal due to flood"),
    ]
                       }
print("--- Starting Graph Stream ---\n")
response = graph.invoke(input_query,config=config)

--- Starting Graph Stream ---



GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}